# Exercise 2: Lookalike letters

*Corpora at Scale · version 0.1.0*

> For Kazakh or Bashkir, count the tokens in a corpus sample that mix Latin and Cyrillic letters. Build a repair table for the 20 most frequent substitutions, and measure how much the repair raises analyser coverage.

**Setup.** `pip install "turkicnlp==0.5.0" pyhfst fasttext-wheel huggingface_hub requests`. Open the notebook from the `exercises/` folder: the first code cell imports the chapter's audit functions from `../code/remaining/corpus.py`. GlotLID (`model_v3.bin`, about 1.6 GB) is downloaded from Hugging Face once; set `GLOTLID_PATH` to use a local copy. The Apertium analysers come with TurkicNLP and are read with the pure-Python `pyhfst`. The text comes from the X-WMT test sets on GitHub, so internet access is needed.

The worked example uses the 1,000 Bashkir sentences of the X-WMT test set: professionally translated news text, so it should contain few errors, which makes every mixed token worth inspecting. Not every mixed token is an error: a Latin brand name followed by a Bashkir case suffix (*Facebookҡа*) is ordinary usage and should not be "repaired". The code separates the two cases with a simple rule (two script runs, each at least two letters long) and repairs only the rest, rewriting minority-script letters in the token's majority script where a letter of the same shape exists.

In [ ]:
import os, re, sys, unicodedata
from collections import Counter
import requests

sys.path.insert(0, os.path.abspath("../code/remaining"))      # the book's corpus.py
from corpus import audit, script_shares, fst_coverage

BASE = "https://raw.githubusercontent.com/turkic-interlingua/til-mt/master/xwmt/test"

def xwmt_side(pair, lang):
    """Non-empty lines of one side of an X-WMT test pair, e.g. xwmt_side("en-ky", "ky")."""
    r = requests.get(f"{BASE}/{pair}/{pair}.{lang}", timeout=30)
    r.raise_for_status()
    return [line.strip() for line in r.text.splitlines() if line.strip()]

def glotlid_model():
    """Unrestricted GlotLID v3 (fastText). Uses GLOTLID_PATH if set, otherwise downloads the model."""
    import fasttext
    path = os.environ.get("GLOTLID_PATH")
    if not path:
        from huggingface_hub import hf_hub_download
        path = hf_hub_download("cis-lmu/glotlid", "model_v3.bin")
    return fasttext.load_model(path)

def lid_labels(model, text, k=1):
    """Labels, best first. A list input avoids a NumPy 2 incompatibility in fasttext's single-string path."""
    return model.predict([text.replace("\n", " ")], k=k)[0][0]

def load_analyser(lang, script="Cyrl"):
    """Apertium analyser from TurkicNLP, read with pyhfst. lookup(word) -> [(reading, weight), ...]."""
    import pyhfst, turkicnlp
    from turkicnlp.resources.registry import ModelRegistry
    turkicnlp.download(lang, processors=["morph"])
    path = ModelRegistry.get_model_path(lang, "morph", "apertium", script=script) / f"{lang}.automorf.hfst"
    fst = pyhfst.HfstInputStream(str(path)).read()
    def lookup(word):
        readings = fst.lookup(word) or fst.lookup(word.lower())
        return [(reading, float(weight)) for reading, weight in readings]
    return lookup

WORD = re.compile(r"[^\W\d_]+")
def words(text):
    return WORD.findall(unicodedata.normalize("NFC", text))

In [ ]:
LOOKALIKE = {  # Cyrillic letter -> Latin letter of the same shape
    "а": "a", "е": "e", "о": "o", "р": "p", "с": "c", "у": "y", "х": "x", "і": "i", "ј": "j", "һ": "h",
    "А": "A", "В": "B", "Е": "E", "К": "K", "М": "M", "Н": "H", "О": "O", "Р": "P", "С": "C", "Т": "T",
    "Х": "X", "У": "Y", "І": "I"}
TO_CYRILLIC = {lat: cyr for cyr, lat in LOOKALIKE.items()}

def script_of(ch):
    name = unicodedata.name(ch, "")
    return "Latin" if "LATIN" in name else "Cyrillic" if "CYRILLIC" in name else "other"

def runs(token):
    """Script runs of the letters in a token, e.g. 'Uberҙың' -> [('Latin', 'Uber'), ('Cyrillic', 'ҙың')]."""
    out = []
    for ch in token:
        if not ch.isalpha():
            continue
        s = script_of(ch)
        if out and out[-1][0] == s:
            out[-1] = (s, out[-1][1] + ch)
        else:
            out.append((s, ch))
    return out

def classify(token):
    r = runs(token)
    if len(r) == 2 and r[0][0] == "Latin" and r[1][0] == "Cyrillic" and min(len(r[0][1]), len(r[1][1])) >= 2:
        return "Latin name + Cyrillic suffix"
    return "lookalike substitution"

def repair(token):
    """Rewrite minority-script letters in the majority script where a same-shape letter exists."""
    counts = Counter(script_of(c) for c in token if c.isalpha())
    major = "Latin" if counts["Latin"] > counts["Cyrillic"] else "Cyrillic"
    table = LOOKALIKE if major == "Latin" else TO_CYRILLIC
    changes, out = [], []
    for c in token:
        if c.isalpha() and script_of(c) != major and c in table:
            changes.append(f"{c} (U+{ord(c):04X}) -> {table[c]}")
            out.append(table[c])
        else:
            out.append(c)
    return "".join(out), changes

def mixed_tokens(lines):
    tokens = [t for line in lines for t in re.findall(r"\w+", line)]
    return tokens, Counter(t for t in tokens if {"Latin", "Cyrillic"} <= {script_of(c) for c in t if c.isalpha()})

In [ ]:
ba = xwmt_side("en-ba", "ba")
tokens, mixed = mixed_tokens(ba)
print(f"Bashkir X-WMT: {len(ba)} sentences, {len(tokens)} tokens, {sum(mixed.values())} mixed-script tokens")
substitutions = Counter()
lookup = load_analyser("bak")
recognised = lambda w: fst_coverage([w], lookup)["accepted_words"] == 1   # <unk> guesses do not count
for tok, n in mixed.most_common():
    kind = classify(tok)
    fixed, changes = repair(tok) if kind == "lookalike substitution" else (tok, [])
    substitutions.update(changes)
    print(f"{tok:14} x{n}  {kind:30} -> {fixed:12} recognised before/after: {recognised(tok)}/{recognised(fixed)}")
print("Substitutions:", substitutions.most_common(20))

In [ ]:
for pair, lang in [("en-kk", "kk"), ("en-ky", "ky"), ("en-kaa", "kaa")]:
    lines = xwmt_side(pair, lang)
    tokens, mixed = mixed_tokens(lines)
    kinds = Counter(classify(t) for t in mixed.elements())
    latin_in_cyrillic = Counter(c for t in mixed.elements() for script, run in runs(t)
                                if script == "Latin" and len(run) == 1 for c in run)
    print(f"{lang}: {len(tokens)} tokens, {sum(mixed.values())} mixed; {dict(kinds)}; "
          f"single Latin letters inside Cyrillic words: {latin_in_cyrillic.most_common(3)}")

## Worked solution

Output on 18 October 2026, UTC:

```
Bashkir X-WMT: 1000 sentences, 20689 tokens, 7 mixed-script tokens
Weezу          x1  lookalike substitution         -> Weezy        recognised before/after: False/False
Uberҙың        x1  Latin name + Cyrillic suffix   -> Uberҙың      recognised before/after: False/False
RTГ            x1  lookalike substitution         -> RTГ          recognised before/after: False/False
Facebookҡа     x1  Latin name + Cyrillic suffix   -> Facebookҡа   recognised before/after: False/False
Тoyota         x1  lookalike substitution         -> Toyota       recognised before/after: False/False
Propertу       x1  lookalike substitution         -> Property     recognised before/after: False/False
CEО            x1  lookalike substitution         -> CEO          recognised before/after: False/False
Substitutions: [('у (U+0443) -> y', 2), ('Т (U+0422) -> T', 1), ('О (U+041E) -> O', 1)]
```

```
kk: 12242 tokens, 1 mixed; {'lookalike substitution': 1}; single Latin letters inside Cyrillic words: []
ky: 9017 tokens, 18 mixed; {'Latin name + Cyrillic suffix': 18}; single Latin letters inside Cyrillic words: []
kaa: 5632 tokens, 628 mixed; {'lookalike substitution': 628}; single Latin letters inside Cyrillic words: [('í', 820)]
```

Of 20,689 tokens in the Bashkir test set, 7 mix Latin and Cyrillic letters. Two are a Latin name followed by a Bashkir suffix (*Uberҙың*, *Facebookҡа*): ordinary usage, which the code leaves unchanged. The other five are substitutions inside Latin names: Cyrillic *Т* in *Тoyota*, Cyrillic *у* in *Weezу* and *Propertу*, Cyrillic *О* in *CEО*. Four of them can be repaired with the same-shape table; *RTГ* cannot, since *Г* has no Latin twin. The repair does not change analyser coverage here, because the repaired forms are English names that the Bashkir analyser does not contain. (It returns guesses such as `Weezy<np><unk>` for capitalised Latin words; the chapter's `fst_coverage` does not count `<unk>` readings as recognised.) The table therefore has four entries, not twenty: an edited test set contains too few substitutions to build one, and the 20-entry table has to come from a web sample.

Kazakh (1 mixed token in 12,242) and Kyrgyz (18, all name plus suffix) look similar. The Karakalpak test set is different: 628 of its 5,632 tokens contain the Latin letter *í* inside a Cyrillic word (820 occurrences, e.g. *салíқ*, *жíлíна*), apparently in place of Cyrillic *ы*. A table built from letter shapes would not find this substitution, since *í* and *ы* look different; it shows up only when single Latin letters inside Cyrillic words are counted. TurkicNLP's Apertium analyser for Karakalpak covers the Latin script only, so coverage before and after a repair could not be measured here.

## Your turn — extension

1. Run the same code on a uniform sample of a web subset (FineWeb2 `bak_Cyrl` or `kaz_Cyrl`), where substitutions are expected to be more frequent than in edited test sets, and build the 20-entry table from that sample.
2. Measure analyser coverage on the *whole* sample before and after repair, not only on the mixed tokens, and report both numbers with the sample size.
3. Kazakh *і* (U+0456) and Latin *i* have the same shape. Find out which way the substitution goes in your sample before you decide which direction to repair.